<a href="https://colab.research.google.com/github/julianocmachado/uci-motion-rnn/blob/main/notebooks/01_exploracao%20/02_visualizacao_dos_sinais_e_atividades.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fase 1 — Notebook 2: visualização dos sinais e das atividades

Neste notebook, vamos explorar visualmente os dados do **usuário 1** da base *Smartphone-Based Recognition of Human Activities and Postural Transitions*.

Ao final, seremos capazes de:

- visualizar separadamente acelerômetro e giroscópio;
- relacionar os sinais às atividades realizadas;
- comparar os experimentos 1 e 2;
- analisar a duração e a distribuição das classes;
- identificar os intervalos sem rótulo, representados por `y = 0`;
- observar com mais detalhe uma transição postural.

> Continuamos trabalhando com sinais contínuos. Ainda não faremos normalização, criação de janelas ou treinamento de redes neurais.

## 1. Preparação do ambiente

O Notebook 1 apresentou detalhadamente o carregamento. Para que este notebook possa ser executado de forma independente no Google Colab, repetiremos de maneira compacta essa preparação.

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

## 2. Parâmetros e nomes das atividades

A base foi adquirida a **50 Hz**. Portanto, cada segundo contém 50 amostras. As classes 1 a 6 são atividades básicas e as classes 7 a 12 são transições posturais. A classe 0 será usada apenas para representar amostras não incluídas em `labels.txt`.

In [3]:
PASTA_BASE = Path('/content/drive/MyDrive/UCI-Motion-Database')
PASTA_RAW = PASTA_BASE / 'RawData'
FREQUENCIA_AMOSTRAGEM = 50  # Hz
USUARIO = 1

NOMES_CARACTERISTICAS = [
    'acc_x', 'acc_y', 'acc_z',
    'gyro_x', 'gyro_y', 'gyro_z'
]

NOMES_ATIVIDADES = {
    0: 'Sem rótulo',
    1: 'Caminhando',
    2: 'Subindo escadas',
    3: 'Descendo escadas',
    4: 'Sentado',
    5: 'Em pé',
    6: 'Deitado',
    7: 'Em pé → sentado',
    8: 'Sentado → em pé',
    9: 'Sentado → deitado',
    10: 'Deitado → sentado',
    11: 'Em pé → deitado',
    12: 'Deitado → em pé'
}

CORES_ATIVIDADES = plt.cm.tab20(np.linspace(0, 1, 13))
CORES_ATIVIDADES[0] = [0.85, 0.85, 0.85, 1.0]

if not PASTA_RAW.exists():
    raise FileNotFoundError(
        'A pasta RawData não foi encontrada. Verifique o caminho da base.'
    )

## 3. Carregamento dos dados do usuário 1

A função abaixo é a mesma lógica construída no Notebook 1: une acelerômetro e giroscópio e converte os segmentos de `labels.txt` em um rótulo por amostra.

In [4]:
NOMES_COLUNAS = [
    'experimento', 'usuario', 'atividade', 'inicio', 'fim'
]

rotulos = pd.read_csv(
    PASTA_RAW / 'labels.txt',
    sep=r'\s+',
    names=NOMES_COLUNAS
)

def carregar_experimento(
    pasta_raw, numero_experimento, numero_usuario, tabela_rotulos
):
    nome_acc = (
        f'acc_exp{numero_experimento:02d}'
        f'_user{numero_usuario:02d}.txt'
    )
    nome_gyro = (
        f'gyro_exp{numero_experimento:02d}'
        f'_user{numero_usuario:02d}.txt'
    )

    acelerometro = np.loadtxt(pasta_raw / nome_acc, dtype=np.float32)
    giroscopio = np.loadtxt(pasta_raw / nome_gyro, dtype=np.float32)

    if acelerometro.shape != giroscopio.shape:
        raise ValueError(
            'Acelerômetro e giroscópio possuem formatos diferentes.'
        )

    X_experimento = np.column_stack((acelerometro, giroscopio))
    y_experimento = np.zeros(len(X_experimento), dtype=np.int32)

    segmentos = tabela_rotulos.loc[
        (tabela_rotulos['experimento'] == numero_experimento)
        & (tabela_rotulos['usuario'] == numero_usuario)
    ]

    for segmento in segmentos.itertuples(index=False):
        inicio_python = int(segmento.inicio) - 1
        fim_python = int(segmento.fim)
        y_experimento[inicio_python:fim_python] = int(segmento.atividade)

    return X_experimento, y_experimento

In [ ]:
rotulos_usuario = rotulos.loc[rotulos['usuario'] == USUARIO].copy()
experimentos = sorted(rotulos_usuario['experimento'].unique())

X_lista = []
y_lista = []
id_experimento_lista = []

for experimento in experimentos:
    X_exp, y_exp = carregar_experimento(
        PASTA_RAW, experimento, USUARIO, rotulos
    )
    X_lista.append(X_exp)
    y_lista.append(y_exp)
    id_experimento_lista.append(
        np.full(len(X_exp), experimento, dtype=np.int32)
    )

X = np.concatenate(X_lista, axis=0)
y = np.concatenate(y_lista, axis=0)
id_experimento = np.concatenate(id_experimento_lista, axis=0)

print('Experimentos:', experimentos)
print('Formato de X:', X.shape)
print('Formato de y:', y.shape)

## 4. Função auxiliar para selecionar um experimento

Os experimentos foram mantidos separados em listas. Isso permite construir um eixo de tempo que começa em zero para cada experimento.

In [ ]:
def selecionar_experimento(numero_experimento):
    indice = experimentos.index(numero_experimento)
    X_exp = X_lista[indice]
    y_exp = y_lista[indice]
    tempo_exp = np.arange(len(X_exp)) / FREQUENCIA_AMOSTRAGEM
    return X_exp, y_exp, tempo_exp

EXPERIMENTO_ESCOLHIDO = 1
X_exp, y_exp, tempo_exp = selecionar_experimento(
    EXPERIMENTO_ESCOLHIDO
)

print(f'Experimento {EXPERIMENTO_ESCOLHIDO:02d}')
print(f'Duração: {tempo_exp[-1]:.2f} s')
print('Formato:', X_exp.shape)

Para visualizar o outro experimento, basta alterar `EXPERIMENTO_ESCOLHIDO` para `2` e executar novamente as células seguintes.

## 5. Visualização do acelerômetro

Os três primeiros canais de `X_exp` correspondem aos eixos do acelerômetro. Cada eixo é apresentado em um gráfico separado para evitar sobreposição excessiva.

In [ ]:
fig, eixos = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
nomes_eixos = ['Eixo x', 'Eixo y', 'Eixo z']
cores = ['tab:blue', 'tab:orange', 'tab:green']

for indice, eixo in enumerate(eixos):
    eixo.plot(tempo_exp, X_exp[:, indice], color=cores[indice], lw=0.8)
    eixo.set_ylabel(nomes_eixos[indice])
    eixo.grid(alpha=0.3)

eixos[0].set_title(
    f'Acelerômetro — usuário {USUARIO}, '    f'experimento {EXPERIMENTO_ESCOLHIDO:02d}'
)
eixos[-1].set_xlabel('Tempo (s)')
fig.supylabel('Aceleração (g)', x=0.02)
plt.tight_layout()
plt.show()

## 6. Visualização do giroscópio

Os três últimos canais representam a velocidade angular medida pelo giroscópio.

In [ ]:
fig, eixos = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for indice, eixo in enumerate(eixos):
    canal = indice + 3
    eixo.plot(tempo_exp, X_exp[:, canal], color=cores[indice], lw=0.8)
    eixo.set_ylabel(nomes_eixos[indice])
    eixo.grid(alpha=0.3)

eixos[0].set_title(
    f'Giroscópio — usuário {USUARIO}, '    f'experimento {EXPERIMENTO_ESCOLHIDO:02d}'
)
eixos[-1].set_xlabel('Tempo (s)')
fig.supylabel('Velocidade angular (rad/s)', x=0.02)
plt.tight_layout()
plt.show()

## 7. Atividades ao longo do tempo

Agora representaremos a classe atribuída a cada amostra. Mudanças abruptas no gráfico correspondem ao início de uma nova atividade ou a um intervalo sem rótulo.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

ax.step(tempo_exp, y_exp, where='post', color='black', lw=0.8)
ax.set_yticks(range(13))
ax.set_yticklabels([NOMES_ATIVIDADES[i] for i in range(13)])
ax.set_xlabel('Tempo (s)')
ax.set_ylabel('Atividade')
ax.set_title(
    f'Rótulos — usuário {USUARIO}, '    f'experimento {EXPERIMENTO_ESCOLHIDO:02d}'
)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Sinais e atividades no mesmo eixo temporal

Uma forma mais informativa de relacionar os dados aos rótulos é usar uma faixa colorida abaixo dos sinais. Neste exemplo, mostraremos o acelerômetro; o mesmo procedimento poderia ser aplicado ao giroscópio.

In [ ]:
mapa_cores = ListedColormap(CORES_ATIVIDADES)
norma_cores = BoundaryNorm(np.arange(-0.5, 13.5, 1), 13)

fig, (ax_sinal, ax_classe) = plt.subplots(
    2, 1, figsize=(14, 6), sharex=True,
    gridspec_kw={'height_ratios': [5, 1]}
)

for indice, nome in enumerate(['acc_x', 'acc_y', 'acc_z']):
    ax_sinal.plot(tempo_exp, X_exp[:, indice], lw=0.8, label=nome)

ax_sinal.set_ylabel('Aceleração (g)')
ax_sinal.set_title('Acelerômetro e atividade correspondente')
ax_sinal.legend(ncol=3)
ax_sinal.grid(alpha=0.3)

ax_classe.imshow(
    y_exp[np.newaxis, :],
    aspect='auto',
    cmap=mapa_cores,
    norm=norma_cores,
    extent=[tempo_exp[0], tempo_exp[-1], 0, 1],
    interpolation='nearest'
)
ax_classe.set_yticks([])
ax_classe.set_ylabel('Classe')
ax_classe.set_xlabel('Tempo (s)')

plt.tight_layout()
plt.show()

A faixa cinza representa `y = 0`. As demais cores identificam as atividades de 1 a 12. Como existem muitas classes, uma tabela será mais legível do que uma legenda extensa nesse gráfico.

In [ ]:
pd.DataFrame({
    'classe': list(NOMES_ATIVIDADES.keys()),
    'atividade': list(NOMES_ATIVIDADES.values())
})

## 9. Distribuição das atividades

A contagem de amostras pode ser convertida diretamente em duração dividindo por 50 Hz. A tabela e o gráfico abaixo consideram os dois experimentos do usuário 1.

In [ ]:
classes, quantidades = np.unique(y, return_counts=True)

tabela_distribuicao = pd.DataFrame({
    'classe': classes,
    'atividade': [NOMES_ATIVIDADES[c] for c in classes],
    'quantidade_amostras': quantidades,
    'duracao_segundos': quantidades / FREQUENCIA_AMOSTRAGEM,
    'percentual': 100 * quantidades / len(y)
})

tabela_distribuicao.round({'duracao_segundos': 2, 'percentual': 2})

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.barh(
    tabela_distribuicao['atividade'],
    tabela_distribuicao['duracao_segundos'],
    color=[CORES_ATIVIDADES[c] for c in classes]
)
ax.set_xlabel('Duração total (s)')
ax.set_ylabel('Atividade')
ax.set_title('Duração das classes — usuário 1')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

plt.tight_layout()
plt.show()

As transições tendem a possuir duração total menor que as atividades básicas. Isso caracteriza um desbalanceamento que deverá ser considerado posteriormente na criação do dataset e na avaliação dos modelos.

## 10. Comparação entre os experimentos 1 e 2

Para comparar experimentos com durações diferentes, apresentaremos o percentual de tempo atribuído a cada classe. Cada linha da tabela soma aproximadamente 100%.

In [ ]:
distribuicao_experimentos = []

for experimento, y_atual in zip(experimentos, y_lista):
    contagens = np.bincount(y_atual, minlength=13)
    percentuais = 100 * contagens / len(y_atual)

    for classe, percentual in enumerate(percentuais):
        distribuicao_experimentos.append({
            'experimento': experimento,
            'classe': classe,
            'atividade': NOMES_ATIVIDADES[classe],
            'percentual': percentual
        })

df_comparacao = pd.DataFrame(distribuicao_experimentos)
tabela_comparacao = df_comparacao.pivot(
    index='experimento', columns='atividade', values='percentual'
)

tabela_comparacao.round(2)

In [ ]:
fig, eixos = plt.subplots(
    len(experimentos), 1, figsize=(14, 7), sharex=True
)

for eixo, experimento, y_atual in zip(eixos, experimentos, y_lista):
    contagens = np.bincount(y_atual, minlength=13)
    percentuais = 100 * contagens / len(y_atual)
    eixo.bar(
        range(13), percentuais, color=CORES_ATIVIDADES, width=0.85
    )
    eixo.set_ylabel(f'Exp. {experimento:02d} (%)')
    eixo.grid(axis='y', alpha=0.3)

eixos[0].set_title('Distribuição percentual por experimento')
eixos[-1].set_xticks(range(13))
eixos[-1].set_xticklabels(
    [NOMES_ATIVIDADES[i] for i in range(13)],
    rotation=45, ha='right'
)
plt.tight_layout()
plt.show()

## 11. Identificação dos segmentos contínuos

Além da quantidade total de amostras, é útil descobrir onde cada trecho começa e termina. A função abaixo transforma o vetor `y` em uma tabela de segmentos contínuos.

In [ ]:
def encontrar_segmentos(y_sinal, frequencia):
    limites = np.flatnonzero(np.diff(y_sinal) != 0) + 1
    inicios = np.r_[0, limites]
    fins_exclusivos = np.r_[limites, len(y_sinal)]

    segmentos = pd.DataFrame({
        'inicio_amostra': inicios,
        'fim_amostra': fins_exclusivos - 1,
        'classe': y_sinal[inicios],
        'quantidade_amostras': fins_exclusivos - inicios
    })

    segmentos['atividade'] = segmentos['classe'].map(NOMES_ATIVIDADES)
    segmentos['inicio_segundos'] = inicios / frequencia
    segmentos['fim_segundos'] = (fins_exclusivos - 1) / frequencia
    segmentos['duracao_segundos'] = (
        segmentos['quantidade_amostras'] / frequencia
    )

    return segmentos

segmentos_exp = encontrar_segmentos(
    y_exp, FREQUENCIA_AMOSTRAGEM
)
segmentos_exp.head(15).round(2)

## 12. Análise dos intervalos sem rótulo

A classe 0 não descreve uma atividade adicional. Ela indica que aquele intervalo não foi incluído nas anotações de `labels.txt`. Vamos listar esses intervalos e calcular quanto representam do experimento.

In [ ]:
segmentos_sem_rotulo = segmentos_exp.loc[
    segmentos_exp['classe'] == 0
].copy()

duracao_sem_rotulo = segmentos_sem_rotulo['duracao_segundos'].sum()
percentual_sem_rotulo = 100 * np.mean(y_exp == 0)

print(f'Quantidade de intervalos sem rótulo: {len(segmentos_sem_rotulo)}')
print(f'Duração total sem rótulo: {duracao_sem_rotulo:.2f} s')
print(f'Percentual do experimento: {percentual_sem_rotulo:.2f}%')

segmentos_sem_rotulo[[
    'inicio_amostra', 'fim_amostra',
    'inicio_segundos', 'fim_segundos', 'duracao_segundos'
]].round(2)

Essas regiões não devem ser automaticamente interpretadas como repouso ou como uma décima terceira atividade. No notebook de preparação do dataset, decidiremos se elas serão removidas ou utilizadas apenas como separação entre segmentos.

## 13. Ampliação de uma transição postural

As transições são breves e ficam difíceis de observar no experimento completo. Selecionaremos automaticamente a primeira transição disponível e mostraremos uma margem de cinco segundos antes e depois do segmento.

In [ ]:
TRANSICOES = set(range(7, 13))
segmentos_transicao = segmentos_exp.loc[
    segmentos_exp['classe'].isin(TRANSICOES)
]

if segmentos_transicao.empty:
    print('Nenhuma transição foi encontrada neste experimento.')
else:
    transicao = segmentos_transicao.iloc[0]
    margem = 5 * FREQUENCIA_AMOSTRAGEM
    inicio = max(0, int(transicao['inicio_amostra']) - margem)
    fim = min(len(y_exp), int(transicao['fim_amostra']) + margem + 1)
    tempo_zoom = np.arange(inicio, fim) / FREQUENCIA_AMOSTRAGEM

    fig, (ax_acc, ax_gyro, ax_classe) = plt.subplots(
        3, 1, figsize=(14, 8), sharex=True,
        gridspec_kw={'height_ratios': [3, 3, 1]}
    )

    for indice, nome in enumerate(['acc_x', 'acc_y', 'acc_z']):
        ax_acc.plot(tempo_zoom, X_exp[inicio:fim, indice], label=nome)

    for indice, nome in enumerate(['gyro_x', 'gyro_y', 'gyro_z']):
        ax_gyro.plot(
            tempo_zoom, X_exp[inicio:fim, indice + 3], label=nome
        )

    ax_acc.set_ylabel('Aceleração (g)')
    ax_gyro.set_ylabel('Vel. angular (rad/s)')
    ax_acc.legend(ncol=3)
    ax_gyro.legend(ncol=3)
    ax_acc.grid(alpha=0.3)
    ax_gyro.grid(alpha=0.3)

    ax_classe.imshow(
        y_exp[inicio:fim][np.newaxis, :],
        aspect='auto', cmap=mapa_cores, norm=norma_cores,
        extent=[tempo_zoom[0], tempo_zoom[-1], 0, 1],
        interpolation='nearest'
    )
    ax_classe.set_yticks([])
    ax_classe.set_ylabel('Classe')
    ax_classe.set_xlabel('Tempo (s)')

    fig.suptitle(
        f"Transição: {NOMES_ATIVIDADES[int(transicao['classe'])]}"
    )
    plt.tight_layout()
    plt.show()

### Questões para análise

1. Quais atividades apresentam maior periodicidade nos sinais?
2. É mais fácil diferenciar visualmente atividades estáticas ou dinâmicas?
3. Quais canais parecem responder mais intensamente à transição selecionada?
4. Por que as transições podem ser mais difíceis de classificar?
5. Que problema surgiria se tratássemos todas as amostras com `y = 0` como uma atividade?
6. A distribuição das classes é semelhante nos dois experimentos?

## 14. Conclusão

Neste notebook:

- visualizamos os seis sinais do usuário 1;
- relacionamos os sinais às atividades anotadas;
- calculamos a duração e a distribuição das classes;
- comparamos os experimentos 1 e 2;
- identificamos os segmentos contínuos e as regiões sem rótulo;
- ampliamos uma transição postural para observar seu comportamento.

No Notebook 3, automatizaremos o carregamento dos demais usuários e organizaremos os metadados necessários para construir o dataset completo.